# Tell: train the detector

Fine-tunes `timm` EfficientNet-B0 to flag **AI-generated receipt images**, calibrates it on a validation split and **freezes a threshold** so that at most 5% of real validation receipts are flagged. It wraps `src/train.py`.

> **This project is a prototype and is not financial advice.** A flag means "this image looks AI-generated", not "this claim is fraud". The only suggested action is: *ask for the card statement or original before paying.*

In [ ]:
# 1) Setup: GPU check + installs
import os, sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
try:
    import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
except Exception as e:
    print('torch missing', e)
if IN_COLAB and not __import__('torch').cuda.is_available():
    print('WARNING: no GPU detected; training will be slow.')
if IN_COLAB:
    !pip -q install timm pytesseract imagehash

In [ ]:
# 2) Data: link norm/ and copy splits + src into a working folder
import os, shutil
REPO = os.path.abspath(os.environ.get('TELL_REPO', '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
DATA = os.path.abspath(os.environ.get('TELL_DATA', os.path.join(REPO, 'data')))
WORK = os.path.abspath(os.environ.get('TELL_WORK', os.path.join(REPO, 'nb_work')))
SPLITS = os.environ.get('TELL_SPLITS', 'splits.csv')
assert os.path.isdir(os.path.join(DATA, 'norm')), f'expected {DATA}/norm (run src/prepare.py, or set TELL_DATA)'
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(os.path.join(WORK, 'norm')):
    os.symlink(os.path.join(DATA, 'norm'), os.path.join(WORK, 'norm'))
shutil.copy(os.path.join(DATA, SPLITS), os.path.join(WORK, SPLITS))
shutil.copytree(os.path.join(REPO, 'src'), os.path.join(WORK, 'src'), dirs_exist_ok=True)
print('working folder:', WORK)

In [ ]:
# 3) Config
SMOKE  = bool(int(os.environ.get('TELL_SMOKE', '0')))   # 1 = quick pipeline check, not a result
EPOCHS = 7            # 5-8 is plenty
SIZE   = 512          # input crop size (px)
BATCH  = 32           # lower to 16 if you hit an out-of-memory error
LR     = 3e-4
MAX_FAR = 0.05        # threshold: flag at most 5% of REAL validation images
OUT    = os.path.join(WORK, 'run')
import pandas as pd
df = pd.read_csv(os.path.join(WORK, SPLITS))
print(df.groupby(['source', 'split', 'label']).size().unstack(fill_value=0))
assert (df.label == 1).sum() > 0, 'no AI images in splits.csv'
if 'synth_smoke' in set(df.source): print('!!! splits contain the SYNTHETIC SMOKE-TEST stand-in: this is not a real run !!!')

In [ ]:
# 4) Train + calibrate + freeze
extra = '--smoke' if SMOKE else f'--epochs {EPOCHS} --size {SIZE} --batch {BATCH} --lr {LR} --workers 2 --max-far {MAX_FAR}'
!cd {WORK} && {sys.executable} src/train.py --splits {SPLITS} --norm-dir norm --out {OUT} {extra}

In [ ]:
# 5) Results: val AUROC, recall and false-alarm rate (Wilson 95% CI), per-source table
print(open(os.path.join(OUT, 'report.md')).read())
import json; print(json.dumps(json.load(open(os.path.join(OUT, 'frozen.json'))), indent=1))

In [ ]:
# 6) Keep the outputs
print('model.pt, frozen.json and report.md are in', OUT)

In [ ]:
# 7) OPTIONAL: Grad-CAM (guarded: a failure here does not affect anything above)
try:
    import numpy as np, torch, matplotlib.pyplot as plt, sys
    sys.path.insert(0, os.path.join(WORK, 'src'))
    import tellmodel as TM, gradcam as G
    dev = TM.pick_device()
    m = TM.make_model(pretrained=False); m.load_state_dict(torch.load(os.path.join(OUT, 'model.pt'), map_location='cpu')); m.to(dev).eval()
    va = df[df.split == 'val']
    picks = pd.concat([va[va.label == 1].sample(3, random_state=0), va[va.label == 0].sample(3, random_state=0)])
    fig, ax = plt.subplots(2, 6, figsize=(18, 6))
    for i, r in enumerate(picks.itertuples()):
        im = TM.load_content(os.path.join(WORK, 'norm', r.path), r.cw, r.ch)
        v = TM.eval_views(im, 256 if SMOKE else SIZE)[0:1]
        cam, z = G.gradcam(m, v, dev)
        base = im.crop((0, 0, min(im.size), min(im.size)))
        ax[0, i].imshow(base); ax[0, i].set_title(('AI' if r.label else 'real') + f'  logit {z:.1f}'); ax[0, i].axis('off')
        ax[1, i].imshow(G.overlay(base, cam)); ax[1, i].axis('off')
    plt.tight_layout(); plt.show()
except Exception as e:
    print('Grad-CAM skipped:', repr(e))

In [ ]:
# 8) OPTIONAL: bookkeeper's-checklist and frequency baselines on the same val split (needs tesseract)
RUN_BASELINES = False
if RUN_BASELINES:
    if IN_COLAB:
        !apt-get -qq install -y tesseract-ocr > /dev/null
    !cd {WORK} && {sys.executable} src/baselines.py --splits {SPLITS} --stats none.csv --norm-dir norm --ocr-cache {WORK}/ocr_cache --out {WORK}/baselines_val.json --workers 2

### Reminder
* Numbers above come from the **validation** split, which was also used to choose the epoch, temperature and threshold. They are optimistic. The honest test is the sealed held-out twin set, run once with `src/eval_heldout.py --unseal` after `frozen.json` is committed.
* A flag means "looks AI-generated", not "fraud". This project is a prototype and is not financial advice.